In [1]:
pip install langchain langchain-openai

   ---------------------------------------- 0.0/580.7 kB ? eta -:--:--
   ---------------------------------------- 580.7/580.7 kB 14.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 30.0 MB/s  0:00:00
   ---------------------------------------- 0.0/988.0 kB ? eta -:--:--
   ---------------------------------------- 988.0/988.0 kB 20.3 MB/s  0:00:00

   -------- ------------------------------- 1/5 [tiktoken]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- ----------------------- 2/5 [openai]
   ---------------- -----------------------


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_core.tools import tool

# Classic agent imports
from langchain_classic.agents import (
    create_tool_calling_agent,
    AgentExecutor
)

from langchain_openai import ChatOpenAI

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    PromptTemplate,
    MessagesPlaceholder
)

**1.) langchain_core.tool:**
- Purpose: Create tools that an LLM can call.
- A tool is simply a Python function that you make available to an AI model.

**2.) create_tool_calling_agent:**
- Purpose: Create an agent that can decide which tools to call based on the user's request.
- It connects:
    - Your LLM.
    - Your available tools.
    - Your prompt template.
- It creates the agent. It does not, by itself, execute the complete agent workflow.
- For example:
    User asks: "Multiply 25 by 4."
    The agent can identify that the multiply tool is relevant and prepare a tool call.

**3.) AgentExecutor:**
- Purpose: Execute the agent and manage its tool-calling workflow.
- For example:
    1. Receives the user's question.
    2. Sends the request to the agent.
    3. The agent decides to call multiply.
    4. The executor runs the Python function.
    5. The tool returns 100.
    6. The executor passes the result back to the agent.
    7. The agent produces the final response.
- Think of it this way:
    - create_tool_calling_agent = Creates the decision-making component.
    - AgentExecutor = Runs the decision-making component and its tools.

**4.) from langchain_openai import ChatOpenAI:**
- Purpose: Connect LangChain applications to OpenAI chat models.
- It is a integration package which allows LangChain to communicate with OpenAI models.

**5.) langchain_core.prompts:**
- 5.1) *ChatPromptTemplate:*
    - Purpose: Create structured chat prompts containing different message types.
    - Example:
        ```
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful mathematics assistant."),
            ("human", "{input}")
        ])
        ```
- 5.2) *SystemMessagePromptTemplate:*
    - Purpose: Create reusable system message templates.
    - Example:
        ```
        system_prompt = SystemMessagePromptTemplate.from_template(
            "You are an expert {domain} assistant."
        )
        ```
- 5.3) *PromptTemplate:*
    - Purpose: Create reusable text-based prompts with dynamic variables.
    - Example:
        ```
        prompt = PromptTemplate.from_template(
            "Explain {topic} in simple language."
        )
        print(prompt.format(topic="Python decorators"))
        ```
    - Output: `Explain Python decorators in simple language.`
- 5.4) *MessagesPlaceholder:*
    - Purpose: Insert a list of messages into a chat prompt dynamically.
    - Example:
        ```
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful assistant."),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        ```

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

euri_api_key = os.getenv("EURI_API_KEY")

print(euri_api_key is not None) # Output: True if the key is loaded correctly, False otherwise

True


In [12]:
# Currently, we are having the data in the form of a dictionary inside our function. 
# In a real-world scenario, this data might come from a database or an API.

# Tool: A tool is simply a Python function that an AI Model or LLM can call to perform a specific task. 
# In this case, we are creating a tool that retrieves the schedule of a course based on its name.

@tool
def get_course_schedule(course_name: str) -> str:
    """
    Get the schedule for a given course.
    """
    schedule = {
        "python" : "Monday and Wednesday, 10:00 AM - 12:00 PM",
        "sql" : "Tuesday and Thursday, 2:00 PM - 4:00 PM",
        "data science" : "Friday, 1:00 PM - 3:00 PM"
    }

    key = course_name.lower()
    return schedule.get(key, "Course not found.") # if the specified key exists, it returns the corresponding value, otherwise it returns the string "Course not found."


- **LangChain's** *@tool* decorator requires a description for your tool.
- If you do not use docstring for a function/tool, then you will get error.
- See the example below:

In [21]:
@tool
def calculate_discounted_price(original_price: float, discounted_percentage: float) -> float:
    discount_amount = (discounted_percentage / 100) * original_price
    discounted_price = original_price - discount_amount
    return round(discounted_price, 2)  # Round to 2 decimal places

ValueError: Function must have a docstring if description not provided.

In [17]:
@tool
def calculate_discounted_price(original_price: float, discounted_percentage: float) -> float:
    """
    Calculate the discounted price given the original price and the discount percentage.
    """
    discount_amount = (discounted_percentage / 100) * original_price
    discounted_price = original_price - discount_amount
    return round(discounted_price, 2)  # Round to 2 decimal places

- If you do not want to use docstrings, you can also provide a description for the tool using the 
`description` parameter in the `@tool` decorator like this:

In [ ]:
@tool(description="It is a real life tool which will help me out for next step recommendation for my learning path.")
def recommend_next_topic(topic: str) -> str:
    topic = topic.lower()
    
    if "python" in topic:
        return "next step for python is to learn web development using Django or Flask."
    elif "data science" in topic:
        return "next step for data science is to learn machine learning algorithms and their applications."
    elif "machine learning" in topic:
        return "next step for machine learning is to learn deep learning and neural networks."
    else:
        return "Sorry, I don't have a recommendation for that topic."